# Synthetic Speech Generation

This notebook generates the AI-generated (fake) speech clips of the project's synthetic test set. Each clip is a sentence from the Kathbath test split, spoken by one of three text-to-speech models:

| Model | Hugging Face ID | How the voice is chosen |
| --- | --- | --- |
| Indic Parler-TTS | `ai4bharat/indic-parler-tts` | A text description of the speaker |
| IndicF5 | `ai4bharat/IndicF5` | Cloned from a short reference recording |
| MMS-TTS | `facebook/mms-tts-hin`, `-tam`, `-tel` | One fixed voice per language |

Each model speaks 720 sentences in each of Hindi, Tamil and Telugu, which gives 2,160 clips per language and 6,480 clips in total.

**Input**

- Kathbath transcripts, downloaded by the notebook. Step 2 turns them into `sentences.csv`: one row per clip, with the clip ID, language, model, source Kathbath recording and speaker, and the sentence text.

**Output**, saved in `/kaggle/working/synthetic_out` and uploaded to a Hugging Face dataset repository:

- `raw/<model>/<language>/<clip_id>.flac`: the generated audio, at the model's own sample rate
- `manifests/<model>/<language>/*.jsonl`: one metadata record per clip

One run of the notebook generates the clips of one model. The two T4 GPUs work in parallel, each on one language at a time. Clips that are already in the Hugging Face repository are skipped, so an interrupted run can simply be started again.

## Requirements

1. **Kaggle session:** Accelerator set to *GPU T4 x2*, Internet switched on.
2. **Hugging Face access:** the account must have accepted the terms of
   - `ai4bharat/indic-parler-tts`
   - `ai4bharat/IndicF5`
   - `ai4bharat/indicvoices_r` (source of the IndicF5 reference voices)
3. **Kaggle secret:** a Hugging Face token with write access, saved as `HF_TOKEN` (Add-ons → Secrets).

## 1. Settings

`GENERATOR` selects the model for this run. `LIMIT` sets how many clips are made per language; a small value such as 5 is useful for a test run, and `None` generates all of them.

In [1]:
GENERATOR = "mms-tts"        # "mms-tts", "indic-parler-tts" or "indicf5"
LANGUAGES = ["hi", "ta", "te"]
LIMIT = None                   # clips per language; None = all 720

HF_REPO = "TarikaHedaoo/dsai-group3-fake-speech"   # Hugging Face dataset repository for the results
UPLOAD = True                # False = keep the results only in /kaggle/working

OUT_DIR = "/kaggle/working/synthetic_out"
BATCH_SIZE = 50              # clips per upload
SEED = 42

## 2. Select the input sentences

The sentences come from the transcripts of Kathbath's **test split**, whose 20 speakers per language do not appear in Kathbath's training data. A sentence is used only if

- it does not also occur in Kathbath's train, validation or known-speaker test transcripts, and
- it has between 6 and 18 words.

From the remaining sentences, 720 per language are drawn with a fixed random seed, and each sentence is assigned to all three models. The result is written to `sentences.csv` (one row per clip). Its MD5 checksum is compared with the expected value, which confirms that every run of the notebook uses exactly the same sentences.

The transcript archive (about 209 MB) is downloaded once per session.

In [2]:
import csv
import hashlib
import os
import random
import tarfile
import urllib.request

TRANSCRIPTS_URL = "https://objectstore.e2enetworks.net/indic-superb/kathbath/clean/transcripts_n2w.tar"
KATHBATH_DIR = "/kaggle/working/kathbath"
SENTENCES_FILE = "/kaggle/working/sentences.csv"
SENTENCES_MD5 = "ad1f2223cb6c68d1f54b249f13386d77"

LANGUAGE_FOLDERS = {"hi": "hindi", "ta": "tamil", "te": "telugu"}
GENERATORS = ["indic-parler-tts", "indicf5", "mms-tts"]
SENTENCES_PER_LANGUAGE = 720
MIN_WORDS, MAX_WORDS = 6, 18
SELECTION_SEED = 42

In [3]:
os.makedirs(KATHBATH_DIR, exist_ok=True)
archive = f"{KATHBATH_DIR}/transcripts_n2w.tar"

if not os.path.exists(archive):
    print("Downloading Kathbath transcripts ...")
    urllib.request.urlretrieve(TRANSCRIPTS_URL, archive)

with tarfile.open(archive) as tar:
    for member in tar.getmembers():
        for folder in LANGUAGE_FOLDERS.values():
            if f"/{folder}/" in member.name:
                tar.extract(member, KATHBATH_DIR)
print("Transcripts ready.")

Transcripts ready.


/tmp/ipykernel_23/164122231.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, KATHBATH_DIR)


In [4]:
def read_transcripts(language_folder, split):
    """Return the rows of one Kathbath transcript file: recording file, speaker, gender and text."""
    path = f"{KATHBATH_DIR}/kb_data_clean_m4a/{language_folder}/{split}/transcription_n2w.txt"
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            kb_file, text = line.rstrip("\n").split("\t")
            # file names look like 844424930324566-536-m.m4a: <recording>-<speaker>-<gender>
            _, speaker, gender = kb_file.replace(".m4a", "").split("-")
            rows.append({"kb_file": kb_file, "kb_speaker": speaker, "kb_gender": gender, "text": text.strip()})
    return rows

In [5]:
rng = random.Random(SELECTION_SEED)
sentence_rows = []

for language, folder in LANGUAGE_FOLDERS.items():
    # sentences that also appear outside the test split are excluded
    other_texts = set()
    for split in ["train", "valid", "test_known"]:
        for row in read_transcripts(folder, split):
            other_texts.add(row["text"])

    # one recording per sentence text, restricted to 6-18 words
    candidates = {}
    for row in read_transcripts(folder, "test"):
        n_words = len(row["text"].split())
        if row["text"] in other_texts or not (MIN_WORDS <= n_words <= MAX_WORDS):
            continue
        if row["text"] not in candidates:
            candidates[row["text"]] = row

    pool = sorted(candidates.values(), key=lambda row: row["kb_file"])
    picked = rng.sample(pool, SENTENCES_PER_LANGUAGE)

    for number, row in enumerate(picked, start=1):
        sent_id = f"{language}_{number:04d}"
        for generator in GENERATORS:
            sentence_rows.append({
                "clip_id": f"{sent_id}_{generator}",
                "sent_id": sent_id,
                "lang": language,
                "generator": generator,
                "split": "test",
                "kb_file": row["kb_file"],
                "kb_speaker": row["kb_speaker"],
                "kb_gender": row["kb_gender"],
                "n_words": len(row["text"].split()),
                "text": row["text"],
            })

columns = ["clip_id", "sent_id", "lang", "generator", "split",
           "kb_file", "kb_speaker", "kb_gender", "n_words", "text"]
with open(SENTENCES_FILE, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(sentence_rows)

with open(SENTENCES_FILE, "rb") as f:
    checksum = hashlib.md5(f.read()).hexdigest()
print(f"{len(sentence_rows)} clips written to {SENTENCES_FILE}")
print("MD5:", checksum)
assert checksum == SENTENCES_MD5, "sentences.csv differs from the expected sentence list"

6480 clips written to /kaggle/working/sentences.csv
MD5: ad1f2223cb6c68d1f54b249f13386d77


## 3. Install libraries

Each model needs a different set of library versions, so only the libraries for the selected model are installed.

In [6]:
if GENERATOR == "mms-tts":
    !pip install -q uroman
elif GENERATOR == "indic-parler-tts":
    !pip install -q git+https://github.com/huggingface/parler-tts.git
elif GENERATOR == "indicf5":
    # The IndicF5 package pins an old numpy that does not exist for this Python version,
    # so it is installed without its pins and the libraries it uses are installed separately.
    !pip install -q --no-deps git+https://github.com/ai4bharat/IndicF5.git
    !pip install -q "transformers<4.50" accelerate cached_path ema_pytorch jieba pydub pypinyin torchdiffeq vocos x_transformers

## 4. Imports and Hugging Face login

In [7]:
import csv
import glob
import hashlib
import json
import os
import threading
import time
from datetime import datetime, timezone

# The model libraries only need PyTorch. Kaggle's TensorFlow does not work with the
# library versions installed above, so transformers must not try to load it.
os.environ["USE_TF"] = "0"

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import CommitOperationAdd, HfApi, login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
api = HfApi(token=hf_token)

if UPLOAD:
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)

print("GPUs available:", torch.cuda.device_count())

GPUs available: 2


## 5. Load the sentence list

In [8]:
with open(SENTENCES_FILE, encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))
for row in all_rows:
    row["n_words"] = int(row["n_words"])

rows_by_language = {}
for language in LANGUAGES:
    rows_by_language[language] = [
        row for row in all_rows
        if row["generator"] == GENERATOR and row["lang"] == language
    ]
    print(f"{GENERATOR} / {language}: {len(rows_by_language[language])} clips")

mms-tts / hi: 720 clips
mms-tts / ta: 720 clips
mms-tts / te: 720 clips


## 6. Find clips that are already finished

A clip counts as finished if its audio file is already in the Hugging Face repository or in the local output folder.

In [9]:
def finished_clip_ids(language):
    folder = f"raw/{GENERATOR}/{language}/"
    done = set()

    for path in glob.glob(f"{OUT_DIR}/{folder}*.flac"):
        done.add(os.path.basename(path).replace(".flac", ""))

    if UPLOAD:
        for path in api.list_repo_files(HF_REPO, repo_type="dataset"):
            if path.startswith(folder):
                done.add(os.path.basename(path).replace(".flac", ""))

    return done

## 7. Text-to-speech models

Each model is wrapped in a small class with the same interface. `generate(row, idx)` returns the waveform and a dictionary describing the voice that was used. `idx` is the position of the clip in the model's list for that language; it decides which voice is used, so voices are spread evenly over the clips.

### 7.1 MMS-TTS

MMS-TTS has one voice per language. Output sample rate: 16 kHz.

In [10]:
class MMSTTS:
    MODEL_IDS = {"hi": "facebook/mms-tts-hin", "ta": "facebook/mms-tts-tam", "te": "facebook/mms-tts-tel"}

    def __init__(self, language, device):
        from transformers import AutoTokenizer, VitsModel

        self.language = language
        self.device = device
        self.model_id = self.MODEL_IDS[language]
        self.model = VitsModel.from_pretrained(self.model_id).to(device).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.sample_rate = self.model.config.sampling_rate

        # Some MMS languages expect romanised input text.
        self.romanizer = None
        if getattr(self.tokenizer, "is_uroman", False):
            import uroman
            self.romanizer = uroman.Uroman()

    def generate(self, row, idx):
        text = row["text"]
        if self.romanizer is not None:
            text = self.romanizer.romanize_string(text)

        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            waveform = self.model(**inputs).waveform[0]

        voice = {"voice_id": f"mms-{self.language}", "voice_gender": "unknown"}
        return waveform.float().cpu().numpy(), voice

### 7.2 Indic Parler-TTS

Indic Parler-TTS takes the sentence and a short description of the speaker. The descriptions rotate over:

- the named speakers recommended for the language on the model card, plus an unnamed male and an unnamed female speaker (for these the model creates a new voice for every clip);
- three speaking paces and three pitch levels.

Every description asks for a clear recording with little background noise. The description used for each clip is saved in its metadata.

In [11]:
class IndicParlerTTS:
    MODEL_ID = "ai4bharat/indic-parler-tts"

    # (speaker name, gender); None means an unnamed speaker
    VOICES = {
        "hi": [("Rohit", "m"), ("Divya", "f"), (None, "m"), (None, "f")],
        "ta": [("Jaya", "f"), (None, "m"), (None, "f")],
        "te": [("Prakash", "m"), ("Lalitha", "f"), (None, "m"), (None, "f")],
    }
    PACES = ["slightly slow", "moderate", "slightly fast"]
    PITCHES = ["low", "moderate", "high"]

    def __init__(self, language, device):
        from parler_tts import ParlerTTSForConditionalGeneration
        from transformers import AutoTokenizer

        self.device = device
        self.model_id = self.MODEL_ID
        self.model = ParlerTTSForConditionalGeneration.from_pretrained(self.MODEL_ID).to(device).eval()
        self.text_tokenizer = AutoTokenizer.from_pretrained(self.MODEL_ID)
        self.description_tokenizer = AutoTokenizer.from_pretrained(self.model.config.text_encoder._name_or_path)
        self.sample_rate = self.model.config.sampling_rate
        self.voices = self.VOICES[language]

    def describe_voice(self, idx):
        n_voices = len(self.voices)
        name, gender = self.voices[idx % n_voices]
        pace = self.PACES[(idx // n_voices) % 3]
        pitch = self.PITCHES[(idx // (n_voices * 3)) % 3]

        if name is not None:
            speaker = name
            voice_id = f"parler-{name.lower()}"
        else:
            speaker = "A male speaker" if gender == "m" else "A female speaker"
            voice_id = f"parler-unnamed-{gender}"

        description = (f"{speaker} speaks at a {pace} pace with a {pitch} pitch, "
                       "in a clear, close-sounding recording with very little background noise.")
        return description, {"voice_id": voice_id, "voice_gender": gender, "description": description}

    def generate(self, row, idx):
        description, voice = self.describe_voice(idx)
        description_ids = self.description_tokenizer(description, return_tensors="pt").to(self.device)
        text_ids = self.text_tokenizer(row["text"], return_tensors="pt").to(self.device)

        with torch.no_grad():
            audio = self.model.generate(
                input_ids=description_ids.input_ids,
                attention_mask=description_ids.attention_mask,
                prompt_input_ids=text_ids.input_ids,
                prompt_attention_mask=text_ids.attention_mask,
            )
        return audio.float().cpu().numpy().squeeze(), voice

### 7.3 IndicF5

IndicF5 copies the voice of a reference recording, and needs the transcript of that recording. The reference recordings come from the test split of IndicVoices-R, a speech dataset released under CC-BY-4.0 for training text-to-speech models. For each language, one recording of 4 to 9 seconds is taken from each of 10 different speakers (5 male, 5 female). Only recordings whose transcript has at least 8 words are used: IndicF5 copies the speaking pace of the reference, and short transcripts are mostly slowly read lists of names or numbers. The clips rotate over these 10 voices. Output sample rate: 24 kHz.

In [12]:
load_lock = threading.Lock()


class IndicF5:
    MODEL_ID = "ai4bharat/IndicF5"
    REFERENCE_DATASET = "ai4bharat/indicvoices_r"
    REFERENCE_CONFIGS = {"hi": "Hindi", "ta": "Tamil", "te": "Telugu"}
    SPEAKERS_PER_GENDER = 5
    MIN_SECONDS, MAX_SECONDS = 4.0, 9.0
    MIN_WORDS = 8  # lists of names or numbers are read slowly, and IndicF5 copies that pace

    def __init__(self, language, device):
        from transformers import AutoModel

        self.model_id = self.MODEL_ID
        self.sample_rate = 24000

        # Both GPUs start the same language at the same time, so they load one after the other.
        with load_lock:
            self.model, loading_info = AutoModel.from_pretrained(
                self.MODEL_ID, trust_remote_code=True, output_loading_info=True)
            if loading_info["missing_keys"]:
                raise RuntimeError(f"IndicF5 weights not loaded: {loading_info['missing_keys'][:3]}")

            # IndicF5 wraps its networks in torch.compile, which is not safe to run from two
            # threads and recompiles for every new clip length. After the weights are loaded,
            # the networks are taken out of the wrapper and used without compiling.
            self.model.ema_model = self.model.ema_model._orig_mod
            self.model.vocoder = self.model.vocoder._orig_mod
            self.model.to(device)

            self.references = self.load_references(language)

    def load_references(self, language):
        from datasets import load_dataset

        folder = f"{OUT_DIR}/references/{language}"
        os.makedirs(folder, exist_ok=True)
        saved_list = f"{folder}/references.json"
        if os.path.exists(saved_list):
            with open(saved_list, encoding="utf-8") as f:
                return json.load(f)

        dataset = load_dataset(self.REFERENCE_DATASET, self.REFERENCE_CONFIGS[language],
                               split="test", streaming=True)
        male, female = [], []
        used_speakers = set()

        for example in dataset:
            speaker = str(example.get("speaker_id") or example.get("speaker"))
            gender = str(example.get("gender", "")).lower()[:1]  # "m" or "f"
            audio = example["audio"]
            seconds = len(audio["array"]) / audio["sampling_rate"]

            if gender not in ("m", "f") or speaker in used_speakers:
                continue
            text = example.get("normalized") or example.get("text")
            if not (self.MIN_SECONDS <= seconds <= self.MAX_SECONDS):
                continue
            if len(text.split()) < self.MIN_WORDS:
                continue
            group = male if gender == "m" else female
            if len(group) >= self.SPEAKERS_PER_GENDER:
                continue

            path = f"{folder}/{speaker}.wav"
            sf.write(path, audio["array"], audio["sampling_rate"])
            group.append({"speaker": speaker, "gender": gender, "path": path,
                          "duration_s": round(seconds, 2),
                          "text": text})
            used_speakers.add(speaker)

            if len(male) == self.SPEAKERS_PER_GENDER and len(female) == self.SPEAKERS_PER_GENDER:
                break

        # Alternate male and female voices.
        references = []
        for m, f in zip(male, female):
            references.append(m)
            references.append(f)

        with open(saved_list, "w", encoding="utf-8") as f:
            json.dump(references, f, ensure_ascii=False, indent=1)
        return references

    def generate(self, row, idx):
        reference = self.references[idx % len(self.references)]
        audio = self.model(row["text"], ref_audio_path=reference["path"], ref_text=reference["text"])

        audio = np.asarray(audio)
        if audio.dtype == np.int16:
            audio = audio.astype(np.float32) / 32768.0

        voice = {"voice_id": f"ivr-{reference['speaker']}", "voice_gender": reference["gender"],
                 "ref_text": reference["text"]}
        return audio.astype(np.float32).squeeze(), voice

In [13]:
MODELS = {"mms-tts": MMSTTS, "indic-parler-tts": IndicParlerTTS, "indicf5": IndicF5}

## 8. Audio quality measures

These values are stored with every clip and are used later in the quality checks:

| Field | Meaning |
| --- | --- |
| `duration_s` | Length in seconds |
| `peak` | Largest absolute sample value |
| `rms` | Average loudness (root mean square) |
| `clipped_frac` | Share of samples at full scale |
| `flag_silent` | RMS below 0.001 |
| `flag_short` | Shorter than 1 second |

In [14]:
def audio_quality(audio, sample_rate):
    if len(audio) == 0:
        return {"duration_s": 0.0, "peak": 0.0, "rms": 0.0, "clipped_frac": 0.0,
                "flag_silent": True, "flag_short": True}

    rms = float(np.sqrt(np.mean(audio ** 2)))
    duration = len(audio) / sample_rate
    return {
        "duration_s": round(duration, 3),
        "peak": round(float(np.max(np.abs(audio))), 4),
        "rms": round(rms, 5),
        "clipped_frac": round(float(np.mean(np.abs(audio) >= 0.999)), 5),
        "flag_silent": rms < 1e-3,
        "flag_short": duration < 1.0,
    }

## 9. Save and upload a batch

After every `BATCH_SIZE` clips, the metadata records of the batch are written to one `.jsonl` file, and the audio files and the metadata file are uploaded to Hugging Face in a single commit.

In [15]:
def save_batch(records, language):
    manifest_path = f"manifests/{GENERATOR}/{language}/{records[-1]['clip_id']}.jsonl"
    os.makedirs(os.path.dirname(f"{OUT_DIR}/{manifest_path}"), exist_ok=True)
    with open(f"{OUT_DIR}/{manifest_path}", "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    if UPLOAD:
        files = [manifest_path] + [r["path"] for r in records if r["error"] is None]
        operations = [CommitOperationAdd(path_in_repo=p, path_or_fileobj=f"{OUT_DIR}/{p}") for p in files]
        api.create_commit(repo_id=HF_REPO, repo_type="dataset", operations=operations,
                          commit_message=f"{GENERATOR}/{language}: {len(records)} clips")

## 10. Generate a list of clips on one GPU

For each clip in the list, the function sets a fixed random seed (`SEED + idx`), generates the audio, saves it as FLAC and records its metadata. A clip that fails is recorded with its error message and no audio file, so it is generated again on the next run.

In [16]:
def generate_clips(language, todo, gpu):
    torch.cuda.set_device(gpu)
    device = f"cuda:{gpu}"
    if not todo:
        return

    model = MODELS[GENERATOR](language, device)
    batch = []

    for count, (idx, row) in enumerate(todo, start=1):
        seed = SEED + idx
        torch.cuda.manual_seed(seed)
        np.random.seed(seed)
        if GENERATOR == "mms-tts":
            torch.manual_seed(seed)  # MMS-TTS also draws random noise on the CPU
        start = time.time()

        try:
            audio, voice = model.generate(row, idx)
            error = None
        except Exception as e:
            audio, voice = np.zeros(0, dtype=np.float32), {}
            error = f"{type(e).__name__}: {e}"
            print(f"[{language}] {row['clip_id']} failed: {error}")

        path = f"raw/{GENERATOR}/{language}/{row['clip_id']}.flac"
        sha1 = None
        if error is None:
            os.makedirs(os.path.dirname(f"{OUT_DIR}/{path}"), exist_ok=True)
            sf.write(f"{OUT_DIR}/{path}", audio, model.sample_rate, format="FLAC")
            with open(f"{OUT_DIR}/{path}", "rb") as f:
                sha1 = hashlib.sha1(f.read()).hexdigest()

        record = dict(row)
        record.update(voice)
        record.update(audio_quality(audio, model.sample_rate))
        record.update({
            "label": "fake",
            "path": path,
            "sha1": sha1,
            "sample_rate": model.sample_rate,
            "model_id": model.model_id,
            "seed": seed,
            "gen_seconds": round(time.time() - start, 2),
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "error": error,
        })
        batch.append(record)

        if len(batch) == BATCH_SIZE or count == len(todo):
            save_batch(batch, language)
            print(f"[{language}] GPU {gpu}: {count}/{len(todo)} done")
            batch = []

## 11. Run on both GPUs

First the list of unfinished clips of every language is made. Each language's list is then split evenly between the GPUs (with two GPUs: GPU 0 takes every first clip, GPU 1 every second clip), so both GPUs have the same amount of work. Both GPUs go through the languages in the same order and run at the same time. MMS-TTS is the exception: it draws part of its random noise on the CPU, which both GPUs would share, so it runs on one GPU only (it takes a few minutes).

In [17]:
worker_errors = []


def gpu_worker(gpu, jobs):
    for language, todo in jobs:
        try:
            generate_clips(language, todo, gpu)
        except Exception as e:
            worker_errors.append(f"{language} on GPU {gpu}: {type(e).__name__}: {e}")
            print(f"[{language}] GPU {gpu} stopped: {type(e).__name__}: {e}")


# MMS-TTS draws part of its random noise on the CPU, which two GPU threads would share.
# It is fast, so it runs on one GPU to keep every clip reproducible.
if GENERATOR == "mms-tts":
    n_gpus = 1
else:
    n_gpus = torch.cuda.device_count()

# The lists are made before any GPU starts, so that both GPUs see the same finished clips.
jobs = {gpu: [] for gpu in range(n_gpus)}
for language in LANGUAGES:
    rows = rows_by_language[language]
    done = finished_clip_ids(language)
    todo = [(idx, row) for idx, row in enumerate(rows) if row["clip_id"] not in done]
    if LIMIT is not None:
        todo = todo[:LIMIT]
    print(f"[{language}] {len(done)} finished, {len(todo)} to generate")
    for gpu in range(n_gpus):
        jobs[gpu].append((language, todo[gpu::n_gpus]))

workers = []
for gpu in range(n_gpus):
    worker = threading.Thread(target=gpu_worker, args=(gpu, jobs[gpu]))
    worker.start()
    workers.append(worker)

for worker in workers:
    worker.join()

if worker_errors:
    raise RuntimeError("Some languages did not finish:\n" + "\n".join(worker_errors))
print("All languages finished.")

[hi] 0 finished, 720 to generate
[ta] 0 finished, 720 to generate
[te] 0 finished, 720 to generate


config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

[te] GPU 0: 700/720 done


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[te] GPU 0: 720/720 done
All languages finished.


## 12. Check the results

Summary of the clips generated in this session, followed by two audio samples per language.

In [18]:
import pandas as pd
from IPython.display import Audio, display

records = []
for path in glob.glob(f"{OUT_DIR}/manifests/{GENERATOR}/*/*.jsonl"):
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

if not records:
    raise SystemExit("No clips were generated in this session (all clips were already finished).")

results = pd.DataFrame(records)
summary = results.groupby("lang").agg(
    clips=("clip_id", "count"),
    failed=("error", "count"),
    silent=("flag_silent", "sum"),
    mean_duration_s=("duration_s", "mean"),
    mean_gen_seconds=("gen_seconds", "mean"),
)
display(summary.round(2))

,clips,failed,silent,mean_duration_s,mean_gen_seconds
lang,,,,,
hi,720,0,0,4.90,0.13
ta,720,0,0,8.07,0.19
te,720,0,0,7.59,0.18


In [19]:
for language in LANGUAGES:
    for path in sorted(glob.glob(f"{OUT_DIR}/raw/{GENERATOR}/{language}/*.flac"))[:2]:
        clip_id = os.path.basename(path).replace(".flac", "")
        text = results.loc[results["clip_id"] == clip_id, "text"].iloc[0]
        print(clip_id, "|", text)
        display(Audio(path))

hi_0001_mms-tts | डियोड्रेंट में कई तरह के सिंथेटिक और रसायन मौजूद होते हैं


hi_0002_mms-tts | आम आदमी पार्टी एक बार फिर सुर्खियों में आ गई है


ta_0001_mms-tts | இதேவேளை உபாதைக் குள்ளாகியிருக்கும் வேகப்பந்து வீச்சாளர் சுரங்க லக்மாலும் இக் குழாமில் பெயரிடப்பட்டுள்ளார்


ta_0002_mms-tts | இந்த கருத்தின்படி செயலாக்க வேகம் புலனுணர்வு கட்டுப்பாடு மற்றும் பணி நினைவகம் ஆகியவை சிந்தனைச் செயலின் முக்கிய கூறுகளாகும்


te_0001_mms-tts | ఈ ప్రాంతంలో ప్రజాజీవితంలో దనసరి అవసరాలకు అనుగుణంగా మారుతూ వస్తుంది


te_0002_mms-tts | ఈ లేఅవుట్లో అత్యంత భారీ బహుళ అంతస్థులతో కూడిన భవనాల నిర్మాణానికి వీలుగా లేఅవుట్ను అభివృద్ధి చేయాలని భావిస్తున్నట్టు సమాచారం
